<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Supervised Learning — Solutions
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

*Worked answers to the **» Predict**, **» Investigate**, **» Modify** and **» Make** tasks in **05_Supervised_Learning**, in the same order. Run the setup cell first; it rebuilds the table and the split as the session did. The numbers quoted are for the course views and `random_state=123`. Where a task asked for a decision, the answer here is one defensible choice, not the only one.*

## Setup

In [ ]:
%pip install pandas_gbq --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas_gbq

# The Google Cloud project that pays for your queries. After the course, use your own.
PROJECT = 'bowyerlab-mimic-teaching'

def query(sql, project=PROJECT):
    """Run a SQL query on BigQuery and return the result as a DataFrame."""
    return pandas_gbq.read_gbq(sql, project_id=project)

# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.patients
df_patients = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.patients`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.admissions
df_admissions = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.admissions`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.icustays
df_icustays = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.icustays`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_vitalsign
df_vitals = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_vitalsign`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.diagnoses_icd
df_diagnoses = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.diagnoses_icd`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_lab
df_labs = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_lab`")

print(len(df_patients), len(df_admissions), len(df_icustays), len(df_vitals), len(df_diagnoses), len(df_labs))
df_stays = df_icustays.merge(df_admissions, on=['subject_id', 'hadm_id'])
df_stays = df_stays.merge(df_patients, on='subject_id')
df_stays = df_stays.merge(df_vitals, on=['subject_id', 'stay_id'])
df_stays = df_stays.merge(df_labs[['stay_id', 'hemoglobin_min', 'hematocrit_min']], on='stay_id', how='left')

df_stays['died'] = df_stays['hospital_expire_flag'].astype(int)
age = df_stays['anchor_age'] + df_stays['admittime'].dt.year - df_stays['anchor_year']
df_stays['age'] = age.astype(int)
time_to_death = df_stays['deathtime'] - df_stays['intime']
df_stays['dead_by_24h'] = time_to_death.dt.total_seconds() < 24 * 60 * 60

# These columns arrive as Decimal objects (BigQuery NUMERIC), which pandas cannot average: make them floats
for column in ['temperature_min', 'temperature_max', 'temperature_mean', 'hemoglobin_min', 'hematocrit_min', 'los']:
    df_stays[column] = df_stays[column].astype(float)

print(df_stays.shape)
# The first ICU stay of each admission. Sorted by stay_id so the split below is the same for everyone
df_model = df_stays.sort_values(['intime', 'stay_id'])
df_model = df_model.drop_duplicates('hadm_id', keep='first')
df_model = df_model.sort_values('stay_id')

grouping = {
    'EW EMER.': 'EMERGENCY', 'DIRECT EMER.': 'EMERGENCY',
    'URGENT': 'URGENT',
    'ELECTIVE': 'ELECTIVE', 'SURGICAL SAME DAY ADMISSION': 'ELECTIVE',
    'OBSERVATION ADMIT': 'OBSERVATION', 'EU OBSERVATION': 'OBSERVATION',
    'DIRECT OBSERVATION': 'OBSERVATION', 'AMBULATORY OBSERVATION': 'OBSERVATION',
}
admission_group = df_model['admission_type'].replace(grouping)
adm_dummies = pd.get_dummies(admission_group, prefix='adm').astype(int)

race_group = df_model['race'].str.replace(r' -.*$', '', regex=True)
race_group = race_group.str.replace(' OR ', '/')
race_group = race_group.replace(['UNKNOWN', 'UNABLE TO OBTAIN', 'PATIENT DECLINED TO ANSWER', 'OTHER'], 'UNKNOWN/OTHER')
race_dummies = pd.get_dummies(race_group, prefix='race').astype(int)

df_model = pd.concat([df_model, adm_dummies, race_dummies], axis=1)
print(df_model.shape)
# Clinical limits are fixed numbers, not learnt from the rows, so they can come before the split
df_model['glucose_max'] = df_model['glucose_max'].mask(df_model['glucose_max'] > 2000)
df_model['heart_rate_max'] = df_model['heart_rate_max'].mask(df_model['heart_rate_max'] > 250)

temperature_gap = df_model['temperature_mean'].isna()
df_model['temperature_missing'] = temperature_gap.astype(int)

is_icd9_diabetes = df_diagnoses['icd_code'].str.startswith('250')
is_icd10_diabetes = df_diagnoses['icd_code'].str.startswith('E10') | df_diagnoses['icd_code'].str.startswith('E11')
df_diabetes = df_diagnoses[is_icd9_diabetes | is_icd10_diabetes]
has_code = df_model['hadm_id'].isin(df_diabetes['hadm_id'])
df_model['diabetes'] = has_code.astype(int)

numeric_features = ['age', 'heart_rate_mean', 'heart_rate_max', 'resp_rate_mean', 'sbp_min', 'glucose_max']
binary_features = ['temperature_missing', 'diabetes'] + list(adm_dummies.columns) + list(race_dummies.columns)
other_columns = ['subject_id', 'anchor_year_group', 'died', 'los', 'dead_by_24h', 'hemoglobin_min', 'hematocrit_min', 'temperature_mean']

df_features = df_model[other_columns + numeric_features + binary_features].copy()
df_features = df_features.dropna(subset=numeric_features)
print(df_features.shape)
print('Stays that ended in death:', round(df_features['died'].mean(), 3))
kept_binary = binary_features.copy()
kept_binary.remove('diabetes')
print(len(kept_binary), 'binary features kept')
from sklearn.model_selection import train_test_split
patients = df_features['subject_id'].unique()
train_patients, test_patients = train_test_split(patients, test_size=0.2, random_state=123)

df_train = df_features[df_features['subject_id'].isin(train_patients)].copy()
df_test = df_features[df_features['subject_id'].isin(test_patients)].copy()

print(len(df_train), 'training rows,', len(df_test), 'test rows')
also_in_train = df_test['subject_id'].isin(df_train['subject_id'])
print(also_in_train.sum(), 'test rows belong to a patient in the training rows')
print('Died, training:', round(df_train['died'].mean(), 3), ' test:', round(df_test['died'].mean(), 3))
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

features = numeric_features + ['temperature_mean'] + kept_binary

prepare = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
prepare.set_output(transform='pandas')
prepare.fit(df_train[features])
X_train = prepare.transform(df_train[features])
X_test = prepare.transform(df_test[features])
y_train = df_train['died']
y_test = df_test['died']

print('Mean age, training rows:', round(df_train['age'].mean(), 2))
print('Mean age, test rows:    ', round(df_test['age'].mean(), 2))
print('Mean age of the test rows after scaling:', round(X_test['age'].mean(), 3))
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(),
)

## Regression

### Investigate: the rule of three
The slope is **0.33** and the intercept about 0: haemoglobin is a third of haematocrit, as the rule says. The model found it from the rows alone.

### Investigate: which rows?
The model learnt from all the rows, and the RMSE, 0.59 g/dL against 2.28 for predicting the mean, is computed on those same rows. It says how well the line fits patients it has already seen. The next patient through theatre is not one of them, so this number cannot tell you how the model will do for them: for that you need rows the model never saw.

## Testing on patients it has not seen

### Investigate: splitting the rows
**One patient's admissions can land on both sides.** 607 of 1,683 test rows, 36%, belong to a patient who is also in the training rows. The test set is meant to stand for new patients, and those rows are not new: the model has seen the same person's age and long-term illness. Split the patients, because that is the claim you want to make.

It happens not to inflate the result on this table: the AUC is 0.796 with the row split and 0.801 with the patient split, a gap well inside the spread between splits in section 7. These features do not identify a person to a logistic regression. A flexible model, or a table with more patient-specific columns, can be a different story, and the patient split costs nothing.

In [ ]:
rows_train, rows_test = train_test_split(df_features, test_size=0.2, random_state=123)

pipeline.fit(rows_train[features], rows_train['died'])
prob_rows = pipeline.predict_proba(rows_test[features])[:, 1]
print('AUC, row split:', round(roc_auc_score(rows_test['died'], prob_rows), 3))   # about 0.796

pipeline.fit(df_train[features], df_train['died'])
prob_patients = pipeline.predict_proba(df_test[features])[:, 1]
print('AUC, patient split:', round(roc_auc_score(df_test['died'], prob_patients), 3))   # about 0.80

### Investigate: the gap
On the test patients the RMSE is still **0.59 g/dL**, against 2.26 for predicting the training mean: no visible gap. A straight line has two numbers to learn from about 6,700 rows, so it cannot bend to fit individual patients the way Model C did, and there is little to overfit. The gap appears with flexible models and few rows: in the practice stretch, a decision tree with no depth limit scores an AUC of 1.0 on its training rows and 0.60 on the test rows.

The test set is still needed. Without it you would not know that the gap was small.

## Preprocessing pipelines

### Investigate: one patient at a time
One patient has no spread, and their own mean is just their own value, so every feature would scale to 0 and the model would see the same blank patient every time. The model has to carry fixed numbers: the training rows' medians, means and spreads, which `fit` stored in the pipeline. That is why preparation is fitted once, on the training rows, and only applied afterwards.

Here, fitting the pipeline on all the rows instead gives the same AUC to four places (0.8014): with 8,412 rows, a mean hardly moves when 20% of them are added. The rule is not about this table's numbers. It is about what a deployed model can know, and in small tables, or for steps that learn more from the rows, such as choosing features, the difference is real.

### Investigate: why diabetes fails
**`diabetes`.** It comes from `diagnoses_icd`, and the codes are assigned after discharge, so they do not exist at 24 hours. It makes almost no difference here (AUC 0.790 with or without it, averaged over 30 splits), and it still goes: a column that passes by doing no harm has not passed.

## Classification

### Predict: against the baseline
**Barely above:** 0.892 against 0.882 for "always survives" on these test rows. Session 4 promised that accuracy could not separate a useful model from a useless one here, and it barely does.

### Investigate: the confusion matrix
197 patients in the test rows died. The model flagged **24** of them and missed **173**. It flagged only 32 people in all. Nobody should switch this on as it stands, and accuracy gave no hint of that.

### Investigate: moving the threshold
| Threshold | Sensitivity | PPV | Accuracy | Flagged per 100 |
| --: | --: | --: | --: | --: |
| 0.5 | 0.12 | 0.75 | 0.89 | 2 |
| 0.1 | 0.74 | 0.24 | 0.70 | 36 |

Accuracy makes the lower threshold look worst, because it counts a missed death and a false alarm as the same mistake and false alarms are far more common. A clinician would weigh the 74% of deaths caught against reviewing 36 patients in every 100, of whom about one in four is a true flag.

## ROC curves and AUC

### Investigate: the third row
At 0.15: died and flagged **3 of 3**, TPR **1.00**; survived and flagged **5 of 7**, FPR **0.71**.

### Investigate: accuracy and AUC
Accuracy is measured at one threshold, and with 11% of patients dying, a threshold of 0.5 is rarely reached: the median predicted probability is 0.066. Most deaths get a probability that is raised and still under 0.5, so the accuracy hardly moves. AUC ignores the threshold and asks only about order. The model ranks patients well (test 0.80), even though the default threshold makes poor use of that.

The test AUC is slightly **above** the training AUC (0.795). That is the luck of this draw, not a better model: over 30 splits the test AUC averages 0.790 against 0.797 for training. There is little overfitting here, as 25 features and 6,700 rows would suggest.

## Cross-validation

### Investigate: 0.80 against 0.78
Not yet. The same model, with nothing changed but which patients were in the test set, ranges from **0.765 to 0.815** over 30 splits: a spread of 0.05, more than twice the claimed gain. On one test set, 0.80 against 0.78 could be the draw. Before believing it, ask for the comparison repeated over many splits, each model's spread as well as its mean, and the same splits for both models. Cross-validation gives a mean with a spread, on the training rows only: 0.787 here, with folds from 0.753 to 0.802.

## Independent practice

### Part 1, Modify: length of stay
RMSE **4.74 days** against **4.80** for predicting the mean: an improvement of 0.06 days, about an hour and a half. With age alone it is no better than the mean at all.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

# How much better than predicting the mean, in days: 0.06, about an hour and a half
model_los = LinearRegression()
model_los.fit(X_train, df_train['los'])
predicted_los = model_los.predict(X_test)

# The baseline: the training mean, for every test patient
mean_stay = df_train['los'].mean()
mean_los = np.full(len(df_test), mean_stay)
print('Model:              ', round(root_mean_squared_error(df_test['los'], predicted_los), 2), 'days')
print('Predicting the mean:', round(root_mean_squared_error(df_test['los'], mean_los), 2), 'days')

### Part 1, Make: R-squared
R-squared is **0.025**: the model explains about 2.5% of the variation (averaged over 30 splits, about 4.5%). The plot shows why. Predictions run from about -3 days, which is impossible, to 12, while real stays reach over 60. Haemoglobin and haematocrit measure nearly the same thing in the same blood. Length of stay depends on what happens **after** the first 24 hours: complications, operations, beds, discharge planning, and for some patients an early death. Nothing in the table knows any of that. The same steps run without complaint in both cases, which is why you always compare with a baseline.

In [ ]:
# Why harder than haemoglobin: it depends on what happens after the first 24 hours, which the table cannot know
print('R-squared:', round(r2_score(df_test['los'], predicted_los), 3))

plt.scatter(predicted_los, df_test['los'], s=4, alpha=0.4)
plt.xlabel('Predicted ICU length of stay (days)')
plt.ylabel('Actual ICU length of stay (days)')
plt.show()

### Part 2, Modify: adding vital signs
Age alone gives an AUC of **0.61**. Adding `sbp_min` gives 0.67, and `resp_rate_mean` as well gives 0.73. All of `features` gives 0.80. Your two may differ; each should add something, and the first usually adds most.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

chosen = ['age', 'sbp_min', 'resp_rate_mean']

model_chosen = LogisticRegression()
model_chosen.fit(X_train[chosen], df_train['died'])
prob_chosen = model_chosen.predict_proba(X_test[chosen])[:, 1]

print(chosen, 'AUC', round(roc_auc_score(df_test['died'], prob_chosen), 3))

### Part 2, Make: a full evaluation
At the default threshold: 24 true positives, 8 false positives, 173 false negatives and 1,469 true negatives. Sensitivity **0.12**, PPV **0.75**, AUC **0.80**. Better than "always survives" in accuracy by one point, and in the sense that "always survives" has a sensitivity of 0 and an AUC of 0.5. The AUC says there is a usable ranking; the sensitivity says the default threshold does not use it.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# Better than "always survives"? Accuracy by one point; the AUC of 0.80 against 0.5 is the real difference
model_full = LogisticRegression()
model_full.fit(X_train, df_train['died'])
pred_full = model_full.predict(X_test)
prob_full = model_full.predict_proba(X_test)[:, 1]

print(confusion_matrix(df_test['died'], pred_full))
print('Sensitivity:', round(recall_score(df_test['died'], pred_full), 2))
print('PPV:        ', round(precision_score(df_test['died'], pred_full), 2))
print('AUC:        ', round(roc_auc_score(df_test['died'], prob_full), 2))

### Part 3, Modify: choosing a threshold
A threshold of about **0.08** reaches sensitivity 0.81. The team then reviews **43 patients in every 100**, and about **one flag in five** is true (PPV 0.22). Whether to deploy is a judgement. Reviewing nearly half of an ICU is hardly a triage tool, and the Part 5 result says some of these catches were already over. A defensible answer: not at this threshold, and perhaps at 0.2, flagging 17 per 100 with a PPV of 0.36 and catching half the deaths, if the team accepts that.

In [ ]:
# Reviews per 100 patients: 43. About one in five is true. Not at this threshold: it asks the team to review nearly half the ICU
threshold = 0.08
flagged = prob_full >= threshold

print('Sensitivity:    ', round(recall_score(df_test['died'], flagged), 2))
print('PPV:            ', round(precision_score(df_test['died'], flagged), 2))
print('Flagged per 100:', round(100 * flagged.mean()))

### Part 4, Make: still ventilated after surgery
2,018 training rows and 524 test rows: surgical patients still in the ICU at 24 hours. **12%** were still ventilated at 48 hours.

| Features | Test AUC | Cross-validated AUC |
| :-- | --: | --: |
| `features` | 0.72 | 0.70 (folds 0.67 to 0.75) |
| `features` and `vent_24h` | **0.91** | 0.91 (0.89 to 0.92) |
| `vent_24h` alone | 0.87 | 0.87 |

**`vent_24h` adds most, and it is fair:** at 24 hours anyone can see who is on a ventilator, so it passes Session 4's test. It is so strong because ventilation at 48 hours is mostly ventilation at 24 hours that has not ended yet: 55% of the patients ventilated at 24 hours were still ventilated at 48, against 3% of the rest.

**Useful?** At a threshold of 0.3 the model flags 17 patients in every 100, catches 79% of those still ventilated at 48 hours, and 56% of its flags are true. The rule the team already uses, "still ventilated now, so still ventilated tomorrow", flags 18 in every 100, catches 84%, and 55% of its flags are true. The model adds almost nothing to that rule. Compare a model with the obvious clinical rule, not only with "always no".

In [ ]:
# Rebuilds Part 4's tables
df_services = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.services`")
df_ventilation = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.ventilation`")

surgical_services = ['CSURG', 'SURG', 'VSURG', 'NSURG', 'TSURG', 'ORTHO']
services = df_services.merge(df_model[['hadm_id', 'intime']], on='hadm_id')
prediction_time = services['intime'] + pd.Timedelta(hours=24)
known_by_then = services['transfertime'] <= prediction_time
is_surgical = services['curr_service'].isin(surgical_services)
surgical = services[known_by_then & is_surgical]
df_model['surgical'] = df_model['hadm_id'].isin(surgical['hadm_id'])

invasive = df_ventilation[df_ventilation['ventilation_status'].isin(['InvasiveVent', 'Tracheostomy'])]
invasive = invasive.merge(df_model[['stay_id', 'intime']], on='stay_id')
for hours in [24, 48]:
    moment = invasive['intime'] + pd.Timedelta(hours=hours)
    covers_moment = (invasive['starttime'] <= moment) & (invasive['endtime'] >= moment)
    ventilated = invasive[covers_moment]
    df_model[f'vent_{hours}h'] = df_model['stay_id'].isin(ventilated['stay_id']).astype(int)

in_scope = df_model['surgical'] & (df_model['los'] >= 1)
surgery_train = df_train[in_scope.loc[df_train.index]].copy()
surgery_test = df_test[in_scope.loc[df_test.index]].copy()
for table in [surgery_train, surgery_test]:
    table['vent_24h'] = df_model.loc[table.index, 'vent_24h']
    table['vent_48h'] = df_model.loc[table.index, 'vent_48h']
print(len(surgery_train), 'training rows,', len(surgery_test), 'test rows')

In [ ]:
# How much vent_24h adds: AUC 0.72 to 0.91. Fair: it is known at 24 hours.
# Useful: barely more than the rule "still ventilated now, so still ventilated tomorrow"
from sklearn.metrics import precision_score, recall_score

threshold = 0.3
for columns in [features, features + ['vent_24h']]:
    pipeline_vent = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression())
    pipeline_vent.fit(surgery_train[columns], surgery_train['vent_48h'])
    prob_vent = pipeline_vent.predict_proba(surgery_test[columns])[:, 1]
    flagged = prob_vent >= threshold
    print(len(columns), 'features: AUC', round(roc_auc_score(surgery_test['vent_48h'], prob_vent), 3),
          ' sensitivity', round(recall_score(surgery_test['vent_48h'], flagged), 2),
          ' PPV', round(precision_score(surgery_test['vent_48h'], flagged), 2),
          ' flagged per 100', round(100 * flagged.mean()))

rule = surgery_test['vent_24h'] == 1
print('The rule:', ' sensitivity', round(recall_score(surgery_test['vent_48h'], rule), 2),
      ' PPV', round(precision_score(surgery_test['vent_48h'], rule), 2),
      ' flagged per 100', round(100 * rule.mean()))

### Part 5, Modify: patients who were already dead
33 test rows go, and the AUC falls from **0.80** to **0.78** (about 0.018 on average over 30 splits). The **0.78** is the honest number for a model used at 24 hours. The other one includes patients whose outcome was already known, with the extreme vital signs of the hours before death, which are easy to rank highly. This is a bigger effect than either the patient split or the scaler, and it comes from the definition of the table, not from the code. Patients discharged from the ICU within the day are still in the table, and their first-day values also cover less than a day: a study would state that decision in its cohort definition.

In [ ]:
# What happens to the AUC: it falls from about 0.80 to 0.78. The honest number at 24 hours is 0.78
rows_train_kept = df_train[df_train['dead_by_24h'] == False]
rows_test_kept = df_test[df_test['dead_by_24h'] == False]

pipeline_rows = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression())
pipeline_rows.fit(rows_train_kept[features], rows_train_kept['died'])
prob_rows = pipeline_rows.predict_proba(rows_test_kept[features])[:, 1]

print(len(rows_test_kept), 'test rows, AUC', round(roc_auc_score(rows_test_kept['died'], prob_rows), 3))

### Part 6, Modify: comparing models
| Model | Mean AUC | Spread between folds |
| :-- | --: | --: |
| Logistic regression | 0.787 | 0.018 |
| KNN, 50 neighbours | 0.777 | 0.016 |
| SVM | 0.727 | 0.037 |
| Decision tree, depth 5 | 0.716 | 0.023 |

**Logistic regression:** it has the highest mean and is the simplest to explain. Its gap over KNN is smaller than the spread between folds, so that comparison alone would not separate them; the gaps over the SVM and the tree are larger.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Which model: logistic regression. Its gap over KNN is inside the spread between folds; over the SVM and tree it is not
candidates = {
    'Logistic regression': LogisticRegression(),
    'Decision tree': DecisionTreeClassifier(max_depth=5, random_state=0),
    'KNN': KNeighborsClassifier(n_neighbors=50),
    'SVM': SVC(),
}
for name, candidate in candidates.items():
    pipeline_candidate = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), candidate)
    scores = cross_val_score(
        pipeline_candidate,
        df_train[features],
        df_train['died'],
        cv=GroupKFold(n_splits=5),
        groups=df_train['subject_id'],
        scoring='roc_auc',
    )
    print(name, 'mean AUC', round(scores.mean(), 3), ' spread', round(scores.std(), 3))

### Part 6, Modify: an unlimited tree
**Model C.** With `max_depth=None` the training AUC is **1.0** and the test AUC **0.60**: it has memorised the training rows. At depth 5 the two are 0.79 and 0.73, a much smaller gap. Session 4's made-up picture, on real data.

In [ ]:
# Model C: a perfect score on the training rows, and much worse on the test rows
tree = make_pipeline(SimpleImputer(strategy='median'), DecisionTreeClassifier(max_depth=None, random_state=0))
tree.fit(df_train[features], df_train['died'])

prob_tree_train = tree.predict_proba(df_train[features])[:, 1]
prob_tree_test = tree.predict_proba(df_test[features])[:, 1]
print('AUC, training rows:', round(roc_auc_score(df_train['died'], prob_tree_train), 3))
print('AUC, test rows:    ', round(roc_auc_score(df_test['died'], prob_tree_test), 3))

### Part 6, Make: the final test
Logistic regression: test AUC **0.80** against **0.787** in cross-validation, close, and on the lucky side of the range in section 7. At threshold 0.2, sensitivity 0.53 and PPV 0.37. For another hospital, the honest answer is that this says nothing: it is one hospital's patients, with its recording habits. It would need an external validation.

In [ ]:
# How close to the cross-validated AUC: within about 0.01. Another hospital: not known, it needs external validation
final_model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression())
final_model.fit(df_train[features], df_train['died'])
prob_final = final_model.predict_proba(df_test[features])[:, 1]
flagged_final = prob_final >= 0.2

print('AUC:        ', round(roc_auc_score(df_test['died'], prob_final), 3))
print('Sensitivity:', round(recall_score(df_test['died'], flagged_final), 2))
print('PPV:        ', round(precision_score(df_test['died'], flagged_final), 2))

### Modify (stretch): KNN without the scaler
Mean AUC falls from **0.716** to **0.669**. Without scaling, a column with a large range, such as `glucose_max` in the hundreds, dominates the distance, and a one-hot column in 0 or 1 hardly counts: the left panel of Session 4's figure. Logistic regression would hardly notice, because its weights can adapt to each column's units.

In [ ]:
pipeline_knn = make_pipeline(SimpleImputer(strategy='median'), KNeighborsClassifier(n_neighbors=10))
scores = cross_val_score(
    pipeline_knn,
    df_train[features],
    df_train['died'],
    cv=GroupKFold(n_splits=5),
    groups=df_train['subject_id'],
    scoring='roc_auc',
)
print('Mean AUC', round(scores.mean(), 3))   # about 0.67, against 0.72 with the scaler

### Make (stretch): let the model decide
With the 24 first-day columns added, 43 in all, the mean AUC rises from **0.787** to **0.796**, and averaged 0.01 better over 30 splits. So on this table **the colleague is slightly right**: 6,700 rows are plenty for a regularised logistic regression with 43 columns, and it did not overfit. It is a small gain for a table that is harder to explain, with near-duplicate columns whose coefficients cannot be read one at a time. Session 4's advice stands in its last sentence: try both, and measure.

In [ ]:
first_day_columns = [column for column in df_vitals.columns if column not in ['subject_id', 'stay_id']]
wide_features = list(dict.fromkeys(features + first_day_columns))
wide_train = df_model.loc[df_train.index]

scores = cross_val_score(
    pipeline,
    wide_train[wide_features],
    wide_train['died'],
    cv=GroupKFold(n_splits=5),
    groups=wide_train['subject_id'],
    scoring='roc_auc',
)
print(len(wide_features), 'columns, mean AUC', round(scores.mean(), 3))

### Make (stretch): a split in time
AUC **0.804** on patients from 2017 onwards, against 0.787 in cross-validation on the earlier years: no drop. Predicted risk averages **0.124** against an observed **0.111**, so the model has become slightly too pessimistic, which AUC cannot see because it only uses the order. So this does not show drift in ranking. It does not show that a deployed model is safe either: it is one hospital, and retrospective, so it cannot show what happens when an alert changes what clinicians do, or when the records or patients change sharply.

In [ ]:
rows_early = df_features[df_features['anchor_year_group'] < '2017 - 2019']
rows_late = df_features[df_features['anchor_year_group'] >= '2017 - 2019']

pipeline.fit(rows_early[features], rows_early['died'])
prob_late = pipeline.predict_proba(rows_late[features])[:, 1]

print(len(rows_early), 'rows before 2017,', len(rows_late), 'from 2017')
print('AUC:', round(roc_auc_score(rows_late['died'], prob_late), 3))
print('Mean predicted risk:', round(prob_late.mean(), 3), ' observed:', round(rows_late['died'].mean(), 3))

### Make (stretch): a long tail
Mean absolute error improves, from **2.70** to **2.25** days; RMSE does not (4.78 against 4.74). A log transform fits the typical stay better and gives up on the very long ones. RMSE squares the errors, so it is dominated by those long stays; absolute error is not. Which is better depends on whether you care more about the typical patient or about planning for the long stays.

In [ ]:
model_log = LinearRegression()
model_log.fit(X_train, np.log1p(df_train['los']))
predicted_log = np.expm1(model_log.predict(X_test))

print('RMSE, plain:', round(root_mean_squared_error(df_test['los'], predicted_los), 2), ' log:', round(root_mean_squared_error(df_test['los'], predicted_log), 2))
print('MAE, plain: ', round(mean_absolute_error(df_test['los'], predicted_los), 2), ' log:', round(mean_absolute_error(df_test['los'], predicted_log), 2))

## Closing

### The two opening questions
1. **Would you switch on the Part 3 alert?** There is no single right answer, and the split in the room is the point. For: it catches about 80% of deaths, and a review is cheap. Against: 43 flags in every 100 patients, four false for every true one, and alert fatigue means the true flags get ignored too. It depends on what a flag triggers, which is the next slide.
2. **Which AUC in a paper?** **0.78.** The model is used at 24 hours, so patients already dead by then are not ones it could help; counting them, 0.80, flatters it. Say when the prediction is made, and evaluate only the patients it would actually be used on.

### Investigate: choosing what to optimise
One example for each box; the class will find others.

| | Acting is cheap | Acting is costly |
| :-- | :-- | :-- |
| **A miss is costly** | A post-operative deterioration alert that triggers a nurse review, or a rule-out pathway for suspected myocardial infarction. Tune for **sensitivity**, and accept many false alarms: if almost no one is missed, a negative is safe to act on. | Deciding whether to return a patient to theatre. Both errors hurt. A model can support the decision, not make it, and the threshold is a judgement call. |
| **A miss is cheap** | A prompt to review VTE prophylaxis or refer to physiotherapy. The threshold barely matters: tune for **flags per 100**: the workload the team will tolerate. | Part 4's ventilation model, used to cancel tomorrow's elective list. Tune for **PPV**: cancel only when the model is very likely right. |

The same model can sit in different boxes depending on what it triggers. Our death model behind a nurse review is top left; behind a decision to limit treatment it would be top right, and no threshold would make it safe on its own.

### Back to the talk
1. **What 0.63 means.** Take one patient who developed sepsis and one who did not: 63% of the time the model scored the one with sepsis higher. A coin would manage 50%.
2. **The vendor.** On whose patients was that AUC measured, and did they include any like ours? Was the split by patient? When is the prediction made, and is every input known by then? What are the sensitivity, PPV and alerts per 100 at the threshold you would run? What is your plan for checking it after it goes live?

### Investigate: another hospital
Not without checking. Haemoglobin divided by haematocrit is the mean corpuscular haemoglobin concentration (MCHC), and the rule of three holds because the body keeps it close to 33 g/dL. In thalassaemia the red cells are small and pale, the MCHC is low, and the model **overestimates** haemoglobin: by about 0.4 g/dL for the 31 patients in our table with a thalassaemia code. In Boston they are rare, so they barely move the average. Where thalassaemia trait is common, in Mediterranean, Middle Eastern, South Asian and South-East Asian populations, they would be a large share of the patients, and near a transfusion threshold 0.4 g/dL matters.

A test set from the same hospital cannot show this: it has the same mix of patients as the training set. Only data from the new population can.

### Investigate: does our model drift?
`temperature_missing` falls from about **2.5%** of stays before 2017 to **1.0%** in 2020 to 2022: the hospital's recording changed. An AUC would not have warned you: it holds at 0.80 across that change. And a model does not stop needing checks because its AUC held. A hospital should watch its inputs (are the columns distributed as they were?), its calibration (as in section 6: does predicted risk match the observed rate?), and its alert rate, regularly and whenever the record system, the population or the practice changes. These are all things you can check before the outcomes arrive, which is why they come before waiting for the AUC to fall.